# 🏗️ GLB to IFC 변환 파이프라인 데모 (Google Colab / Kaggle)

이 노트북은 프로젝트에서 생성된 3D 모델(GLB/glTF/OBJ 등)을 **BIM 표준 포맷인 IFC (Industry Foundation Classes) IFC4** 형태로 자동으로 변환하는 단일 파이프라인 데모입니다.

### ✨ 주요 기능
1. **지능형 자동 스케일링 (Auto-Scale)**: 입력 메쉬의 바운딩 박스를 계산하여 밀리미터(mm), 센티미터(cm) 단위를 미터(m) 단위의 IFC 표준 좌표계로 자동 변환합니다.
2. **BIM 공간 구조화**: `IfcProject` → `IfcSite` → `IfcBuilding` → `IfcBuildingStorey` 계층을 표준 규격대로 빌드합니다.
3. **메쉬 병합 및 분할 옵션**: 여러 부재 메쉬를 하나의 객체로 자동 병합하거나 개별 물리 부재로 나누어 매핑할 수 있습니다.
4. **의미론 매핑(Semantic Mapping)**: 기본 프록시 부재(`IfcBuildingElementProxy`) 외에도 필요에 따라 `IfcWall`, `IfcDoor`, `IfcSlab` 등 세부 BIM 유형으로 매핑할 수 있습니다.

---
## 1. 라이브러리 설치
3D 기하 데이터 처리를 위한 `trimesh`와 IFC 데이터 구조 핸들링을 위한 `ifcopenshell`을 설치합니다.

In [ ]:
# 필요한 패키지 설치
!pip install trimesh ifcopenshell numpy -q
print("🎉 핵심 라이브러리 설치가 완료되었습니다!")

---
## 2. 변환 파이프라인 스크립트 작성
로컬에 작성된 `glb_to_ifc.py` 변환 코드를 코랩 세션에 생성합니다.

In [ ]:
%%writefile glb_to_ifc.py
import argparse
import sys
import os
import trimesh
import numpy as np
import ifcopenshell
from ifcopenshell.api import run

def auto_detect_scale(mesh):
    """
    Intelligently auto-detect the units of the 3D mesh based on its bounding box extents,
    and return the appropriate scale factor to convert the model to Meters (SI standard for IFC).
    """
    extents = mesh.extents  # [dx, dy, dz] in original units
    max_span = float(max(extents))
    
    # 3D generated meshes often range either:
    # 1. Near unit scale (e.g. max span 0.5 to 5.0) -> already in Meters
    # 2. In Centimeters (e.g. a chair of 100cm high) -> max span 30.0 to 200.0
    # 3. In Millimeters (e.g. a chair of 1000mm high) -> max span 300.0 to 2000.0
    if max_span > 250.0:
        scale = 0.001
        unit_name = "Millimeters (mm)"
    elif max_span > 25.0:
        scale = 0.01
        unit_name = "Centimeters (cm)"
    else:
        scale = 1.0
        unit_name = "Meters (m)"
        
    print(f"[Auto-Scale] Mesh bounding box extents: {extents}")
    print(f"[Auto-Scale] Max span is {max_span:.3f}. Auto-detected unit: {unit_name}.")
    print(f"[Auto-Scale] Applying scale factor: {scale:.4f} (converting to meters)")
    return scale

def hollow_mesh(mesh, thickness=0.1):
    """
    Hollows out a solid watertight mesh by creating an inward-offset inner shell 
    with inverted normals, and combining it with the outer shell.
    """
    if not isinstance(mesh, trimesh.Trimesh):
        return mesh
    
    print(f"[*] Hollowing solid mesh with wall thickness: {thickness}m...")
    
    # 1. Create the inner shell by moving vertices inwards along their normals
    inner = mesh.copy()
    normals = inner.vertex_normals
    
    # Move vertices inwards
    inner.vertices = inner.vertices - (thickness * normals)
    
    # 2. Invert face orientation of the inner mesh
    inner.invert()
    
    # 3. Concatenate outer and inner meshes
    hollowed = trimesh.util.concatenate([mesh, inner])
    return hollowed

def convert_glb_to_ifc(
    glb_path,
    ifc_path,
    ifc_class="IfcBuildingElementProxy",
    project_name="3D to IFC Project",
    site_name="Default Site",
    building_name="Default Building",
    storey_name="Default Storey",
    element_name="Converted Mesh",
    scale_factor=None,  # None means 'auto'
    merge_meshes=True,
    hollow=False,
    thickness=0.1
):
    """
    Converts a GLB/glTF/OBJ model to a structured IFC4 file.
    """
    if not os.path.exists(glb_path):
        raise FileNotFoundError(f"Input GLB file not found: {glb_path}")

    print(f"[*] Loading 3D model from: {glb_path}...")
    scene = trimesh.load(glb_path)
    
    # Extract geometries from the loaded scene/mesh
    meshes_to_convert = []
    
    if isinstance(scene, trimesh.Scene):
        if len(scene.geometry) == 0:
            raise ValueError("No geometries found in the GLB scene.")
        
        if merge_meshes:
            print("[*] Merging multiple meshes in the GLB into a single unified mesh...")
            # scene.dump(concatenate=True) combines all geometries into a single Trimesh
            merged_mesh = scene.dump(concatenate=True)
            if isinstance(merged_mesh, list):
                # If dump returns a list of meshes, concatenate them manually
                merged_mesh = trimesh.util.concatenate(merged_mesh)
            meshes_to_convert.append((element_name, merged_mesh))
        else:
            print(f"[*] Processing {len(scene.geometry)} meshes separately...")
            for name, geom in scene.geometry.items():
                if isinstance(geom, trimesh.Trimesh):
                    # trimesh scene.graph stores the transform matrices for each node
                    try:
                        transform = scene.graph.get(name)[0]
                    except Exception:
                        transform = np.identity(4)
                    
                    # Create a copy and apply the transform to get world/scene-space coordinates
                    geom_copy = geom.copy()
                    geom_copy.apply_transform(transform)
                    meshes_to_convert.append((name, geom_copy))
    elif isinstance(scene, trimesh.Trimesh):
        meshes_to_convert.append((element_name, scene))
    else:
        raise ValueError(f"Unsupported geometry type loaded: {type(scene)}")
    
    # 2. Compute Scaling Factor
    # If the scale factor is set to 'auto' or None, we compute it from the bounding box of the overall model
    if scale_factor is None or str(scale_factor).lower() == "auto":
        # Combine all meshes to get the global bounding box for scale detection
        if len(meshes_to_convert) == 1:
            detected_scale = auto_detect_scale(meshes_to_convert[0][1])
        else:
            combined_mesh = trimesh.util.concatenate([m[1] for m in meshes_to_convert])
            detected_scale = auto_detect_scale(combined_mesh)
        actual_scale = detected_scale
    else:
        try:
            actual_scale = float(scale_factor)
            print(f"[Manual-Scale] Using user-specified scale factor: {actual_scale:.6f}")
        except ValueError:
            print(f"[Warning] Invalid scale factor specified: '{scale_factor}'. Defaulting to auto-detection.")
            combined_mesh = trimesh.util.concatenate([m[1] for m in meshes_to_convert])
            actual_scale = auto_detect_scale(combined_mesh)

    # 3. Create a clean IFC file (IFC4 schema)
    print("[*] Creating new IFC4 model...")
    model = run("project.create_file", version="IFC4")
    
    # Create the standard BIM spatial structure (Project must exist before assigning units)
    project = run("root.create_entity", model, ifc_class="IfcProject", name=project_name)
    
    # Assign default SI units (meters for length, square meters for area, cubic meters for volume)
    length_unit = run("unit.add_si_unit", model, unit_type="LENGTHUNIT", prefix=None)
    area_unit = run("unit.add_si_unit", model, unit_type="AREAUNIT")
    volume_unit = run("unit.add_si_unit", model, unit_type="VOLUMEUNIT")
    run("unit.assign_unit", model, units=[length_unit, area_unit, volume_unit])
    
    # Create the rest of the spatial structure
    site = run("root.create_entity", model, ifc_class="IfcSite", name=site_name)
    building = run("root.create_entity", model, ifc_class="IfcBuilding", name=building_name)
    storey = run("root.create_entity", model, ifc_class="IfcBuildingStorey", name=storey_name)
    
    # Aggregate structural elements: Project contains Site, Site contains Building, Building contains Storey
    run("aggregate.assign_object", model, relating_object=project, products=[site])
    run("aggregate.assign_object", model, relating_object=site, products=[building])
    run("aggregate.assign_object", model, relating_object=building, products=[storey])
    
    # Add geometric representation contexts
    context = run("context.add_context", model, context_type="Model")
    body_context = run(
        "context.add_context", 
        model, 
        context_type="Model", 
        context_identifier="Body", 
        target_view="MODEL_VIEW", 
        parent=context
    )
    
    # 4. Generate IFC geometry representations
    for name, mesh in meshes_to_convert:
        num_verts = len(mesh.vertices)
        num_faces = len(mesh.faces)
        print(f"[*] Processing element '{name}': {num_verts} vertices, {num_faces} faces")
        
        if num_verts == 0 or num_faces == 0:
            print(f"[Warning] Skipping empty mesh '{name}'.")
            continue
            
        # Create a working copy of the mesh and scale it to meters
        scaled_mesh = mesh.copy()
        scaled_mesh.vertices = scaled_mesh.vertices * actual_scale
        
        # Apply hollowing if requested
        if hollow:
            scaled_mesh = hollow_mesh(scaled_mesh, thickness=thickness)
            num_verts = len(scaled_mesh.vertices)
            num_faces = len(scaled_mesh.faces)
            print(f"[*] Hollowing applied. New structure: {num_verts} vertices, {num_faces} faces")
            
        # Convert vertices and faces for IFC
        raw_verts = scaled_mesh.vertices.tolist()
        scaled_verts = [tuple(float(c) for c in v) for v in raw_verts]
        vertices_list = [scaled_verts]
        
        # Convert faces (0-based indices)
        raw_faces = scaled_mesh.faces.tolist()
        faces_list = [tuple(int(idx) for idx in f) for f in raw_faces]
        faces_list_wrapped = [faces_list]
        
        # Create IFC Product Entity of the specified class (e.g., IfcBuildingElementProxy)
        try:
            element = run("root.create_entity", model, ifc_class=ifc_class, name=name)
        except Exception as e:
            print(f"[Error] Failed to create IFC entity of class '{ifc_class}': {e}")
            print("Defaulting to 'IfcBuildingElementProxy'.")
            element = run("root.create_entity", model, ifc_class="IfcBuildingElementProxy", name=name)
        
        # Place the physical product inside our Building Storey
        run("spatial.assign_container", model, relating_structure=storey, products=[element])
        
        # Add mesh representation to the body context
        representation = run(
            "geometry.add_mesh_representation",
            model,
            context=body_context,
            vertices=vertices_list,
            faces=faces_list_wrapped
        )
        
        # Associate representation with the physical product
        run("geometry.assign_representation", model, product=element, representation=representation)
        
    # Write completed model to file
    print(f"[*] Writing IFC model to: {ifc_path}...")
    model.write(ifc_path)
    print("[+] Conversion completed successfully!")
    return actual_scale

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Convert GLB/glTF 3D models into IFC4 BIM files.")
    parser.add_argument("glb_path", help="Path to the input GLB or glTF file.")
    parser.add_argument("ifc_path", help="Path to save the output IFC file.")
    parser.add_argument("--class", dest="ifc_class", default="IfcBuildingElementProxy", 
                        help="IFC entity class (e.g. IfcWall, IfcBuildingElementProxy, IfcFurniture). Default: IfcBuildingElementProxy.")
    parser.add_argument("--name", default="Converted Mesh", help="Name of the converted element in the IFC model.")
    parser.add_argument("--scale", default="auto", help="Scaling factor. E.g. '0.001' for mm->m, or 'auto' (default) to auto-detect.")
    parser.add_argument("--no-merge", action="store_true", help="Process sub-meshes as separate elements instead of merging.")
    parser.add_argument("--project", default="3D to IFC Project", help="Name of the IfcProject.")
    parser.add_argument("--site", default="Default Site", help="Name of the IfcSite.")
    parser.add_argument("--building", default="Default Building", help="Name of the IfcBuilding.")
    parser.add_argument("--storey", default="Default Storey", help="Name of the IfcBuildingStorey.")
    parser.add_argument("--hollow", action="store_true", help="Hollow out the solid mesh to leave only walls.")
    parser.add_argument("--thickness", type=float, default=0.1, help="Wall thickness in meters when hollowing is enabled. Default: 0.1.")

    args = parser.parse_args()
    
    try:
        convert_glb_to_ifc(
            glb_path=args.glb_path,
            ifc_path=args.ifc_path,
            ifc_class=args.ifc_class,
            project_name=args.project,
            site_name=args.site,
            building_name=args.building,
            storey_name=args.storey,
            element_name=args.name,
            scale_factor=args.scale,
            merge_meshes=not args.no_merge,
            hollow=args.hollow,
            thickness=args.thickness
        )
    except Exception as e:
        print(f"\n[Fatal Error] {e}", file=sys.stderr)
        sys.exit(1)


---
## 3. 테스트용 대형 GLB 모델 생성 (2000mm 크기의 큐브)
자동 스케일링(Auto-Scale) 기능을 검증하기 위해 가상의 **밀리미터(mm) 단위 대형 큐브 모델**(한 변의 길이가 2000mm)을 생성하여 `test_cube.glb` 파일로 저장해 보겠습니다.

In [ ]:
import trimesh
import numpy as np

# 2000mm 크기의 큐브 메시 생성
s = 2000.0 / 2.0  # 반경 1000mm
vertices = np.array([
    [-s, -s, -s], [s, -s, -s], [s, s, -s], [-s, s, -s],
    [-s, -s, s],  [s, -s, s],  [s, s, s],  [-s, s, s]
], dtype=float)

faces = np.array([
    [0, 2, 1], [0, 3, 2],
    [4, 5, 6], [4, 6, 7],
    [0, 1, 5], [0, 5, 4],
    [1, 2, 6], [1, 6, 5],
    [2, 3, 7], [2, 7, 6],
    [3, 0, 4], [3, 4, 7]
], dtype=int)

test_mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
test_mesh.export('test_cube.glb', file_type='glb')
print("💾 테스트용 대형 큐브(test_cube.glb) 모델 생성이 완료되었습니다!")

---
## 4. 파이프라인 실행 (GLB ➡️ IFC4)
CLI 명령어를 실행하여 큐브 모델을 `IfcBuildingElementProxy` 객체로 변환합니다. 
작동 시 지능형 스케일러가 2000.0mm를 감지하여 자동으로 `0.001` 비율을 적용하고 미터(m) 단위로 축소 변환할 것입니다.

In [ ]:
from google.colab import files
try:
    # 변환된 일반 파일과 속비우기 파일 둘 다 다운로드
    files.download('test_cube.ifc')
    files.download('test_cube_hollow.ifc')
    print("💾 브라우저 다운로드 팝업이 활성화되었습니다!")
except Exception as e:
    print(f"[!] 다운로드 에러 (로컬 주피터 환경의 경우 브라우저 팝업 미지원 가능): {e}")


---
## 5. 생성된 IFC 구조 및 자동 스케일 확인 (파이썬 코드 검증)
`ifcopenshell` 모듈을 로드하여 생성된 IFC 파일 내부의 건축 계층 정보 및 좌표 데이터가 2.0m(미터) 규격으로 올바르게 변환되었는지 검사합니다.

In [ ]:
import ifcopenshell

model = ifcopenshell.open("test_cube.ifc")

print("=== IFC 데이터 모델 정보 ===")
projects = model.by_type("IfcProject")
print(f"Project:  {projects[0].Name if projects else 'None'}")

sites = model.by_type("IfcSite")
print(f"Site:     {sites[0].Name if sites else 'None'}")

buildings = model.by_type("IfcBuilding")
print(f"Building: {buildings[0].Name if buildings else 'None'}")

storeys = model.by_type("IfcBuildingStorey")
print(f"Storey:   {storeys[0].Name if storeys else 'None'}")

products = model.by_type("IfcBuildingElementProxy")
print(f"Products: {[p.Name for p in products]}")

print("\n=== 기하 스케일 (Geometry Scale) 검증 ===")
# 메쉬 포인트 리스트 데이터 확인
point_lists = model.by_type("IfcCartesianPointList3D")
if point_lists:
    coords = point_lists[0].CoordList
    max_val = max(max(pt) for pt in coords)
    min_val = min(min(pt) for pt in coords)
    span = max_val - min_val
    print(f"메쉬 버텍스 수:  {len(coords)}개")
    print(f"최소 좌표값:    {min_val:.3f} m")
    print(f"최대 좌표값:    {max_val:.3f} m")
    print(f"측정된 정육면체 한 변 크기: {span:.3f} 미터 (m)")
    
    if abs(span - 2.0) < 0.001:
        print("\n🎉 검증 통과! 원래 2000mm 크기였던 기하정보가 2.0 미터(m) 단위로 완벽하게 자동 변환되었습니다.")
    else:
        print("\n❌ 스케일 에러: 기대 크기는 2.0m이나 다릅니다.")
else:
    print("❌ 3D 지오메트리 데이터를 찾을 수 없습니다.")

---
## 6. 🔥 추가 기능: 메쉬 내부 비우기 (Hollowing/Shelling)
입력 모델이 속이 꽉 찬(Solid) 형태인 경우, 가벼운 구조를 만들기 위해 **안쪽을 비우고 일정 벽 두께만 남겨두는 셸링(Shelling) 기능**을 지원합니다.

### ⚙️ 옵션 안내
- `--hollow`: 메쉬 내부 속비우기를 활성화합니다.
- `--thickness [두께]`: 남겨둘 벽의 두께를 **미터(m)** 단위로 설정합니다. (기본값: `0.1` = 10cm)

In [ ]:
# 1. 내부 비우기 옵션(--hollow) 및 벽 두께 0.15m(15cm)를 지정하여 변환 실행
!python glb_to_ifc.py test_cube.glb test_cube_hollow.ifc --class "IfcBuildingElementProxy" --name "Hollowed Box" --hollow --thickness 0.15

print("\n[+] 내부를 비운 hollowed IFC 파일 변환이 완료되었습니다!")

In [ ]:
# 2. 생성된 Hollowed IFC 파일 내부의 정점(vertex) 수 및 좌표 기하 검증
# 안을 비웠으므로 기존 8개의 정점이 이중벽(겉면 8개 + 안면 8개) 구조가 되어 총 16개의 정점으로 변환됩니다.
model_hollow = ifcopenshell.open("test_cube_hollow.ifc")

print("=== Hollowed IFC 기하 데이터 검증 ===")
point_lists = model_hollow.by_type("IfcCartesianPointList3D")
if point_lists:
    coords = point_lists[0].CoordList
    max_val = max(max(pt) for pt in coords)
    min_val = min(min(pt) for pt in coords)
    span = max_val - min_val
    print(f"이중벽 메쉬 버텍스 수:  {len(coords)}개 (Solid의 8개에서 두 배로 증가)")
    print(f"최소 좌표값:    {min_val:.3f} m")
    print(f"최대 좌표값:    {max_val:.3f} m")
    print(f"측정된 정육면체 크기: {span:.3f} m")
    
    if len(coords) == 16:
        print("\n🎉 검증 통과! 오프셋 셸링 알고리즘이 완벽하게 적용되어 속이 빈 이중벽 구조가 형성되었습니다.")
    else:
        print("\n❌ 검증 실패: 정점 수 확인 필요.")
else:
    print("❌ 기하정보를 찾을 수 없습니다.")

---
## 7. 📥 IFC 결과 파일 다운로드
아래 셀을 실행하면 로컬 PC로 변환 완료된 `test_cube.ifc`와 내부를 비운 `test_cube_hollow.ifc` 파일을 바로 다운로드할 수 있습니다.

In [ ]:
from google.colab import files
try:
    files.download('test_cube.ifc')
    print("💾 브라우저 다운로드 팝업이 활성화되었습니다!")
except Exception as e:
    print(f"[!] 다운로드 에러 (로컬 주피터 환경의 경우 브라우저 팝업 미지원 가능): {e}")